# Day 62：毕业设计开发 Day 4：FastAPI 全量接口与前端控制台联调（Jupyter Notebook 实战手册）

> **所属阶段**：第 6 阶段 · 第六阶段：毕业设计与就业冲刺 ｜ **第 9 周**：星辰智脑毕业设计全栈开发与上线  
> **GitLab 工单**：`STARMIND-062` ｜ **开发分支**：`feat/day62-capstone-fastapi-console`  
> **配套 HTML5 课件**：`courseware/day62/index.html` ｜ **独立版本快照**：`versions/day62/`

---

## 🎙️ 老陈开场旁白

早啊各位！毕业设计 Day 4，后端接口与前端控制台大合龙！今天直接上真家伙——用 FastAPI 把星辰智脑全量 API 全部拉齐：存活/就绪探针 `/healthz`、`/readyz`，同步对话 `/api/v1/chat`，打字机流式推送 `/api/v1/chat/stream`（标准 `text/event-stream` SSE 协议），知识库动态入库 `/api/v1/knowledge/docs`，以及主管审批工作台 `/api/v1/approvals/{ticket_id}`。同时咱们还要写一个前端控制台流式解析与联调客户端，用 `TestClient` 把每一条接口从头到尾跑通！

### 承上启下
- **昨日基线**：昨天我们完成了星辰智脑的多 Agent 编排、MCP 工具与 HITL 人工审批引擎，今天把它暴露为标准 Web API 并完成前后端联调。
- **明日预告**：明天（Day 63）进入毕业设计质量保障阶段：自动化测试、Ragas 量化评估报告、高并发压测与 Token 成本优化。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day62'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-062）

前端管理控制台（Next.js/React）需要一套契约清晰、支持 SSE 打字机流式渲染、支持审批单交互与知识库动态录入的后端 HTTP 接口。

**今日迭代目标：**
- 基于 FastAPI 实现全量生产级 REST + SSE 接口，并挂载全局 `X-Trace-Id` 中间件。
- 实现 SSE 事件流协议（`meta` / `delta` / `done` 事件）与前端流式解析器。
- 通过 FastAPI `TestClient` 完成全接口端到端自动化联调验证。

**验收标准：**
- 所有响应均自动携带 `X-Trace-Id` 响应头；`/healthz` 与 `/readyz` 返回 200。
- `/api/v1/chat/stream` 返回 `text/event-stream` 且前端解析器可完整拼装回复与事件类型。
- 支持动态录入新知识文档并立即检索命中，支持高危报销挂起并通过 `/api/v1/approvals/{ticket_id}` 审批。

---

## 📚 课堂核心技术笔记

### 1. SSE 响应头里的隐藏神器：`X-Accel-Buffering: no`

除了在 Nginx 配置里写 `proxy_buffering off;`，后端 FastAPI 在返回 `StreamingResponse` 时直接加上响应头 `X-Accel-Buffering: no` 与 `Cache-Control: no-cache`，可以由后端单方面通知上游 Nginx 立即透传每一个分块，无需每次发版都找运维改 Nginx 配置。

> **⚠️ 老陈避坑**：SSE 协议中每条事件之间必须用两个换行符 `\n\n` 分隔，少一个换行会导致浏览器 `EventSource` 一直等待不触发回调。

### 2. 为什么流式对话接口常用 POST 而不是原生 EventSource 的 GET？

浏览器原生 `EventSource` 只支持 GET 请求，既不能把长对话上下文和 JWT `Authorization` 放进请求体/请求头，还容易超 URL 长度限制。生产前端普遍用 `fetch()` 发起 POST 请求并读取 `response.body.getReader()` 手动解析 SSE 帧。

> **⚠️ 老陈避坑**：手动解析 SSE 帧时要注意处理 TCP 粘包/半包，按 `\n\n` 切分完整帧后再解析 `event:` 和 `data:`。


## 💻 实操步骤 1：步骤一：实现星辰智脑 FastAPI 全量 REST + SSE 后端服务

- **目标文件**：`versions/day62/src/day62_api_server.py`
- **老陈代码旁白**：使用 FastAPI 构建完整后端：含全局 `X-Trace-Id` 中间件、双探针、知识库动态入库、同步/SSE 流式对话与主管审批工作台接口。


In [ ]:
"""
Day 62 - 模块一：星辰智脑 FastAPI 全量 REST + SSE 后端服务
"""
from __future__ import annotations

import json
import re
import uuid
from typing import Any, Dict, Iterator, List, Optional

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field


class DocCreateRequest(BaseModel):
    doc_id: str = Field(..., min_length=2, max_length=32)
    title: str = Field(..., min_length=2, max_length=100)
    content: str = Field(..., min_length=4, max_length=2000)


class ChatRequest(BaseModel):
    session_id: str = Field(..., min_length=1, max_length=64)
    query: str = Field(..., min_length=1, max_length=1000)
    employee: str = Field(default="张明", max_length=32)


class ApprovalDecisionRequest(BaseModel):
    approve: bool
    approver: str = Field(..., min_length=1, max_length=32)
    note: str = Field(default="", max_length=200)


class StarMindServiceState:
    """轻量可测试的服务内存状态（生产可无缝替换为 SQLAlchemy + Redis）"""

    def __init__(self) -> None:
        self.docs: Dict[str, Dict[str, str]] = {
            "DOC-01": {
                "doc_id": "DOC-01",
                "title": "差旅住宿报销标准",
                "content": "一线城市出差住宿上限650元/晚，需在费用中心上传电子专票核销。",
            },
            "DOC-02": {
                "doc_id": "DOC-02",
                "title": "网关错误码ERR_4291手册",
                "content": "错误码ERR_4291表示部门日Token配额已触发硬熔断，请联系管理员提额。",
            },
        }
        self.tickets: Dict[str, Dict[str, Any]] = {}

    def search_doc(self, query: str) -> Optional[Dict[str, str]]:
        q_low = query.lower()
        best_doc, best_hits = None, 0
        for doc in self.docs.values():
            hay = f"{doc['title']} {doc['content']}".lower()
            hits = sum(1 for ch in set(q_low) if ch.strip() and ch in hay)
            if hits > best_hits:
                best_hits, best_doc = hits, doc
        return best_doc if best_hits >= 2 else None

    def handle_chat(self, req: ChatRequest) -> Dict[str, Any]:
        if any(k in req.query for k in ("报销", "提交")) and re.search(r"\d+", req.query):
            amount = float(re.search(r"(\d+(?:\.\d+)?)", req.query).group(1))
            if amount > 2000:
                tid = f"APR-{uuid.uuid4().hex[:6].upper()}"
                self.tickets[tid] = {
                    "ticket_id": tid,
                    "session_id": req.session_id,
                    "employee": req.employee,
                    "amount": amount,
                    "status": "PENDING",
                }
                return {
                    "session_id": req.session_id,
                    "routed_agent": "action_agent",
                    "status": "PENDING_APPROVAL",
                    "ticket_id": tid,
                    "answer": f"报销金额 {amount:.2f} 元超过2000元上限，已转主管审批（单号 {tid}）。",
                }
            return {
                "session_id": req.session_id,
                "routed_agent": "action_agent",
                "status": "COMPLETED",
                "ticket_id": None,
                "answer": f"报销单已自动提交成功，金额 {amount:.2f} 元。",
            }

        hit = self.search_doc(req.query)
        if hit:
            return {
                "session_id": req.session_id,
                "routed_agent": "knowledge_agent",
                "status": "COMPLETED",
                "ticket_id": None,
                "answer": f"{hit['content']} [{hit['doc_id']}]",
            }
        return {
            "session_id": req.session_id,
            "routed_agent": "knowledge_agent",
            "status": "COMPLETED",
            "ticket_id": None,
            "answer": "知识库未检索到相关条款，为防幻觉暂不作答。",
        }


def create_app() -> FastAPI:
    app = FastAPI(title="StarMind Enterprise AI API", version="1.0.0")
    state = StarMindServiceState()
    app.state.starmind = state

    @app.middleware("http")
    async def trace_id_middleware(request: Request, call_next):
        trace_id = request.headers.get("X-Trace-Id") or f"tr-{uuid.uuid4().hex[:8]}"
        request.state.trace_id = trace_id
        response = await call_next(request)
        response.headers["X-Trace-Id"] = trace_id
        return response

    @app.get("/healthz")
    def healthz() -> Dict[str, str]:
        return {"status": "alive"}

    @app.get("/readyz")
    def readyz() -> Dict[str, Any]:
        return {"status": "ready", "docs_count": len(state.docs)}

    @app.post("/api/v1/knowledge/docs")
    def upsert_doc(req: DocCreateRequest) -> Dict[str, Any]:
        state.docs[req.doc_id] = req.model_dump()
        return {"ok": True, "doc_id": req.doc_id, "total_docs": len(state.docs)}

    @app.post("/api/v1/chat")
    def chat(req: ChatRequest, request: Request) -> Dict[str, Any]:
        res = state.handle_chat(req)
        res["trace_id"] = request.state.trace_id
        return res

    @app.post("/api/v1/chat/stream")
    def chat_stream(req: ChatRequest, request: Request) -> StreamingResponse:
        res = state.handle_chat(req)
        trace_id = request.state.trace_id

        def event_generator() -> Iterator[str]:
            meta = {"trace_id": trace_id, "routed_agent": res["routed_agent"], "status": res["status"]}
            yield f"event: meta\ndata: {json.dumps(meta, ensure_ascii=False)}\n\n"
            text = res["answer"]
            step = 10
            for i in range(0, len(text), step):
                chunk = {"delta": text[i : i + step]}
                yield f"event: delta\ndata: {json.dumps(chunk, ensure_ascii=False)}\n\n"
            done = {"status": res["status"], "ticket_id": res["ticket_id"]}
            yield f"event: done\ndata: {json.dumps(done, ensure_ascii=False)}\n\n"

        return StreamingResponse(
            event_generator(),
            media_type="text/event-stream",
            headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
        )

    @app.post("/api/v1/approvals/{ticket_id}")
    def decide_approval(ticket_id: str, req: ApprovalDecisionRequest) -> Dict[str, Any]:
        if ticket_id not in state.tickets:
            raise HTTPException(status_code=404, detail="审批单不存在")
        ticket = state.tickets[ticket_id]
        if ticket["status"] != "PENDING":
            raise HTTPException(status_code=409, detail="审批单已处理，禁止重复操作")
        ticket["status"] = "APPROVED_EXECUTED" if req.approve else "REJECTED"
        ticket["approver"] = req.approver
        ticket["note"] = req.note
        return ticket

    return app


app = create_app()

if __name__ == "__main__":
    from fastapi.testclient import TestClient

    client = TestClient(app)
    print("健康探针:", client.get("/healthz").json())
    r = client.post("/api/v1/chat", json={"session_id": "s1", "query": "差旅住宿标准"})
    print("同步对话:", r.status_code, r.headers["X-Trace-Id"], r.json()["answer"])


### 🔍 步骤 1 代码深度解析

利用工厂函数 `create_app()` 隔离应用状态便于单元测试；流式接口显式附带 `X-Accel-Buffering: no` 防止 Nginx 缓冲吞掉打字机效果。

**预期运行输出参考：**
```text
健康探针: {'status': 'alive'}
同步对话: 200 tr-... 一线城市出差住宿上限650元/晚，需在费用中心上传电子专票核销。[DOC-01]
```


## 💻 实操步骤 2：步骤二：实现前端控制台 SSE 流式解析器与全接口联调脚本

- **目标文件**：`versions/day62/src/day62_console_client.py`
- **老陈代码旁白**：模拟前端控制台消费 `/api/v1/chat/stream` 的 SSE 事件帧，还原完整打字机文本、路由元信息与审批单号。


In [ ]:
"""
Day 62 - 模块二：前端控制台 SSE 事件解析器与端到端联调客户端
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional


@dataclass
class ParsedSSEMessage:
    meta: Dict[str, Any] = field(default_factory=dict)
    chunks: List[str] = field(default_factory=list)
    done: Dict[str, Any] = field(default_factory=dict)

    @property
    def full_text(self) -> str:
        return "".join(self.chunks)


def parse_sse_stream(raw_sse: str) -> ParsedSSEMessage:
    """解析标准 SSE 文本流（按双换行分帧，提取 event 与 JSON data）"""
    parsed = ParsedSSEMessage()
    frames = [f.strip() for f in raw_sse.strip().split("\n\n") if f.strip()]
    for frame in frames:
        event_name = "message"
        data_lines: List[str] = []
        for line in frame.splitlines():
            if line.startswith("event:"):
                event_name = line.split(":", 1)[1].strip()
            elif line.startswith("data:"):
                data_lines.append(line.split(":", 1)[1].strip())
        if not data_lines:
            continue
        payload = json.loads("\n".join(data_lines))
        if event_name == "meta":
            parsed.meta = payload
        elif event_name == "delta":
            parsed.chunks.append(payload.get("delta", ""))
        elif event_name == "done":
            parsed.done = payload
    return parsed


if __name__ == "__main__":
    from fastapi.testclient import TestClient
    from src.day62_api_server import create_app

    client = TestClient(create_app())
    resp = client.post("/api/v1/chat/stream", json={"session_id": "fe-1", "query": "报错 ERR_4291 怎么办"})
    msg = parse_sse_stream(resp.text)
    print("SSE 元信息:", msg.meta)
    print("分片数量:", len(msg.chunks))
    print("完整回复:", msg.full_text)


### 🔍 步骤 2 代码深度解析

`parse_sse_stream` 与浏览器端 `fetch + ReadableStream` 解析逻辑完全一致，可在 CI 阶段自动验证后端 SSE 分帧格式是否符合前端控制台预期。

**预期运行输出参考：**
```text
SSE 元信息: {'trace_id': 'tr-...', 'routed_agent': 'knowledge_agent', 'status': 'COMPLETED'}
分片数量: 5
完整回复: 错误码ERR_4291表示部门日Token配额已触发硬熔断，请联系管理员提额。[DOC-02]
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day62/tests/test_day62.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 62 单元测试：FastAPI 全量 REST/SSE 接口与前端解析联调"""
import sys
from pathlib import Path
from fastapi.testclient import TestClient

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day62_api_server import create_app
from src.day62_console_client import parse_sse_stream


def test_health_and_trace_header():
    client = TestClient(create_app())
    r1 = client.get("/healthz")
    r2 = client.get("/readyz", headers={"X-Trace-Id": "tr-custom-99"})
    assert r1.status_code == 200 and r1.headers["X-Trace-Id"].startswith("tr-")
    assert r2.status_code == 200 and r2.headers["X-Trace-Id"] == "tr-custom-99"
    assert r2.json()["docs_count"] == 2


def test_dynamic_doc_upsert_and_sse_stream():
    client = TestClient(create_app())
    up = client.post(
        "/api/v1/knowledge/docs",
        json={"doc_id": "DOC-09", "title": "年假政策", "content": "入职满一年享受10天带薪年假。"},
    )
    assert up.status_code == 200 and up.json()["total_docs"] == 3

    stream_resp = client.post("/api/v1/chat/stream", json={"session_id": "s9", "query": "年假有几天？"})
    assert stream_resp.status_code == 200
    assert "text/event-stream" in stream_resp.headers["content-type"]
    assert stream_resp.headers["X-Accel-Buffering"] == "no"

    parsed = parse_sse_stream(stream_resp.text)
    assert "[DOC-09]" in parsed.full_text
    assert len(parsed.chunks) >= 2
    assert parsed.done["status"] == "COMPLETED"


def test_approval_workflow_and_conflict_guard():
    client = TestClient(create_app())
    r = client.post("/api/v1/chat", json={"session_id": "s10", "query": "提交采购报销 6800 元"})
    body = r.json()
    assert body["status"] == "PENDING_APPROVAL"
    tid = body["ticket_id"]

    ok = client.post(f"/api/v1/approvals/{tid}", json={"approve": True, "approver": "王总", "note": "同意"})
    assert ok.status_code == 200 and ok.json()["status"] == "APPROVED_EXECUTED"

    dup = client.post(f"/api/v1/approvals/{tid}", json={"approve": True, "approver": "王总"})
    assert dup.status_code == 409

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：为 FastAPI 服务增加请求参数校验失败测试用例

**题目要求：**
使用 `TestClient` 验证当 `query` 为空字符串时，`/api/v1/chat` 自动返回 HTTP 422 状态码。

**思路提示：** 调用 `client.post('/api/v1/chat', json={'session_id': 's1', 'query': ''})` 并断言 `status_code == 422`。


In [ ]:
from fastapi.testclient import TestClient
from src.day62_api_server import create_app

client = TestClient(create_app())
resp = client.post("/api/v1/chat", json={"session_id": "s1", "query": ""})
assert resp.status_code == 422
print("空提问 422 校验拦截通过")


**答案解析与评分标准：**

Pydantic `min_length=1` 在入口层直接挡掉非法空请求，避免浪费下游 Embedding 与 LLM Token。


### 作业 2（企业实战）：企业实战：给审批工作台增加待审批列表查询接口

**题目要求：**
编写过滤函数 `list_pending_tickets(tickets_dict)`，仅返回 `status == 'PENDING'` 的待审批工单并按金额倒序排列。

**思路提示：** 用列表推导过滤后按 `key=lambda x: x['amount'], reverse=True` 排序。


In [ ]:
def list_pending_tickets(tickets: dict) -> list[dict]:
    pending = [t for t in tickets.values() if t["status"] == "PENDING"]
    return sorted(pending, key=lambda x: x["amount"], reverse=True)


sample = {
    "T1": {"ticket_id": "T1", "amount": 3000, "status": "PENDING"},
    "T2": {"ticket_id": "T2", "amount": 9000, "status": "PENDING"},
    "T3": {"ticket_id": "T3", "amount": 5000, "status": "REJECTED"},
}
res = list_pending_tickets(sample)
assert [x["ticket_id"] for x in res] == ["T2", "T1"]
print("待审批高额优先队列通过:", res)


**答案解析与评分标准：**

审批工作台优先展示高金额待办能显著提升财务与主管审批效率。


### 作业 3（架构进阶）：架构进阶：实现接口级 CORS 白名单校验器

**题目要求：**
编写 `is_allowed_origin(origin, allowlist)`，禁止生产环境使用 `*` 放行全部来源，仅允许白名单内的精确域名。

**思路提示：** 若 `allowlist` 包含 `'*'` 则抛出 `ValueError`，否则判断 `origin in allowlist`。


In [ ]:
def is_allowed_origin(origin: str, allowlist: list[str]) -> bool:
    if "*" in allowlist:
        raise ValueError("生产环境禁止配置通配符 CORS='*'")
    return origin in allowlist


assert is_allowed_origin("https://starmind.corp.cn", ["https://starmind.corp.cn"]) is True
assert is_allowed_origin("https://evil.site", ["https://starmind.corp.cn"]) is False
print("生产 CORS 白名单安全校验通过")


**答案解析与评分标准：**

带 Cookie/凭证的企业控制台若误配 `allow_origins=['*']` 会带来严重的跨站请求风险。


---

## 🎯 老陈收尾复盘

爽！全量 FastAPI 接口、全局 `X-Trace-Id` 响应头、SSE 逐段打字机流、知识库入库与审批工作台全部联调通过！明天（Day 63）咱们给系统上强度：Ragas 量化评估 + 并发压测 + Token 成本优化！
